# **# CELL 1: ENVIRONMENT SETUP & DIRECTORY CONFIGURATION**

Purpose: Mounts Drive, defines paths, creates folders, and verifies GPU/models.


In [ ]:
# ============================================
# CELL 1: SETUP
# Run only this cell every time you open Colab
# ============================================

from google.colab import drive
import os

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("✅ Drive mounted!")
except:
    print("✅ Drive already mounted!")

# Define all paths in one place
DRIVE_BASE   = "/content/drive/MyDrive/BreastCancerNew"
SAVE_DIR     = f"{DRIVE_BASE}/results"
LOCAL_BASE   = "/content"
PROCESSED    = f"{LOCAL_BASE}/BUSI_processed"
DATASET_PATH = f"{LOCAL_BASE}/BUSI_dataset"
BASE_PATH    = f"{LOCAL_BASE}/BUSI_raw/Dataset_BUSI_with_GT"
IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 30
RANDOM_SEED  = 42

# Create permanent folders in Drive
# (No action taken if they already exist)
for folder in [
    f"{DRIVE_BASE}",
    f"{SAVE_DIR}",
    f"{SAVE_DIR}/models",
    f"{SAVE_DIR}/plots",
    f"{SAVE_DIR}/plots/gradcam",
    f"{SAVE_DIR}/history",
]:
    os.makedirs(folder, exist_ok=True)

print("✅ Drive folders ready!")

# Check content in Drive
models_dir = f"{SAVE_DIR}/models"
saved_models = [
    f for f in os.listdir(models_dir)
    if f.endswith('.keras')
]

print(f"\n📊 Currently {len(saved_models)}/10 models are available in Drive:")
all_models = [
    'InceptionV3_feature_extractor',
    'InceptionV3_finetune',
    'Xception_feature_extractor',
    'Xception_finetune',
    'DenseNet121_feature_extractor',
    'DenseNet121_finetune',
    'VGG16_feature_extractor',
    'VGG16_finetune',
    'ResNet50_feature_extractor',
    'ResNet50_finetune',
]

for m in all_models:
    fname = f"{m}_best.keras"
    if fname in saved_models:
        size = os.path.getsize(f"{models_dir}/{fname}")
        print(f"   ✅ {m} ({size/1024/1024:.1f} MB)")
    else:
        print(f"   ❌ {m} — Not yet trained")

import tensorflow as tf
import numpy as np
import random
random.seed(RANDOM_SEED)

print(f"\n✅ TensorFlow version: {tf.__version__}")
print(f"✅ GPU available: {tf.config.list_physical_devices('GPU')}")
print("\n✅ CELL 1 complete — Proceed to the next cell!")

Mounted at /content/drive
✅ Drive mounted!
✅ Drive folders ready!

📊 Currently 6/10 models are available in Drive:
   ✅ InceptionV3_feature_extractor (0.0 MB)
   ✅ InceptionV3_finetune (237.7 MB)
   ✅ Xception_feature_extractor (207.0 MB)
   ✅ Xception_finetune (223.4 MB)
   ✅ DenseNet121_feature_extractor (71.3 MB)
   ✅ DenseNet121_finetune (75.8 MB)
   ❌ VGG16_feature_extractor — Not yet trained
   ❌ VGG16_finetune — Not yet trained
   ❌ ResNet50_feature_extractor — Not yet trained
   ❌ ResNet50_finetune — Not yet trained

✅ TensorFlow version: 2.20.0
✅ GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

✅ CELL 1 complete — Proceed to the next cell!


# **Cell 2 — Dataset Prepare (processed data save in the Drive)**


In [ ]:
# ============================================
# CELL 2: DATASET PREPARATION
# Once the dataset is created, it stays on Drive
# Next time it will only be copied — no download needed
# ============================================

import os, shutil, zipfile, cv2
import numpy as np
from pathlib import Path
import random
random.seed(RANDOM_SEED)

# ============================================
# Save processed images to Drive
# To avoid re-processing if the session disconnects
# ============================================
DRIVE_PROCESSED = f"{DRIVE_BASE}/processed_images"
os.makedirs(f"{DRIVE_PROCESSED}/benign",    exist_ok=True)
os.makedirs(f"{DRIVE_PROCESSED}/malignant", exist_ok=True)

# ============================================
# PREPROCESS FUNCTION
# ============================================
def preprocess_image(img_path, mask_path):
    img  = cv2.imread(str(img_path))
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (img.shape[1], img.shape[0]))
    _, binary_mask = cv2.threshold(
        mask, 127, 255, cv2.THRESH_BINARY)
    masked_img = img.copy()
    masked_img[binary_mask == 0] = 0
    coords = cv2.findNonZero(binary_mask)
    if coords is None:
        cropped = masked_img
    else:
        x, y, w, h = cv2.boundingRect(coords)
        pad_x = int(w * 0.05)
        pad_y = int(h * 0.05)
        x1 = max(0, x - pad_x)
        y1 = max(0, y - pad_y)
        x2 = min(img.shape[1], x + w + pad_x)
        y2 = min(img.shape[0], y + h + pad_y)
        cropped = masked_img[y1:y2, x1:x2]
    h, w = cropped.shape[:2]
    if h != w:
        max_dim = max(h, w)
        square  = np.zeros(
            (max_dim, max_dim, 3), dtype=np.uint8)
        y_s = (max_dim - h) // 2
        x_s = (max_dim - w) // 2
        square[y_s:y_s+h, x_s:x_s+w] = cropped
    else:
        square = cropped
    return cv2.resize(square, (IMG_SIZE, IMG_SIZE))

# ============================================
# Check if processed images exist on Drive
# ============================================
b_drive = len([f for f in os.listdir(
    f"{DRIVE_PROCESSED}/benign")
    if f.endswith('.png')])
m_drive = len([f for f in os.listdir(
    f"{DRIVE_PROCESSED}/malignant")
    if f.endswith('.png')])

print(f"📊 Processed images on Drive:")
print(f"   Benign:    {b_drive} units")
print(f"   Malignant: {m_drive} units")

if b_drive < 430 or m_drive < 200:
    # ============================================
    # Download BUSI Dataset
    # ============================================
    # Skipped if BUSI is already at BASE_PATH (e.g. uploaded by hand to /content/BUSI_raw)
    if not os.path.isdir(BASE_PATH):
        print("\n⏳ Downloading BUSI Dataset...")
        os.makedirs("/content/BUSI_raw", exist_ok=True)
        os.makedirs("/root/.kaggle",     exist_ok=True)

        KAGGLE_JSON = "/content/drive/MyDrive/kaggle.json"
        if not os.path.exists(KAGGLE_JSON):
            raise FileNotFoundError(
                f"BUSI not found at {BASE_PATH} and no Kaggle key at {KAGGLE_JSON}. "
                "Either put kaggle.json in MyDrive, or download BUSI "
                "(Dataset_BUSI_with_GT) and place it at BASE_PATH.")
        shutil.copy(KAGGLE_JSON, "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)

        import subprocess
        subprocess.run(
            ["kaggle", "datasets", "download",
             "-d", "aryashah2k/breast-ultrasound-images-dataset",
             "-p", "/content/BUSI_raw", "--force"],
            check=True)
        with zipfile.ZipFile(
            "/content/BUSI_raw/"
            "breast-ultrasound-images-dataset.zip"
        ) as z:
            z.extractall("/content/BUSI_raw")
        print("✅ Download complete!")

    # ============================================
    # Preprocess + Save to Drive
    # ============================================
    print("\n⏳ Preprocessing + Saving to Drive...")
    for cat in ['benign', 'malignant']:
        folder = Path(BASE_PATH) / cat
        images = sorted([
            f for f in folder.iterdir()
            if f.suffix == '.png' and 'mask' not in f.name
        ])
        count = 0
        for img_path in images:
            mask_path = folder / img_path.name.replace(
                '.png', '_mask.png')
            if not mask_path.exists():
                continue
            processed = preprocess_image(img_path, mask_path)
            # Save to Drive (permanent)
            cv2.imwrite(
                f"{DRIVE_PROCESSED}/{cat}/{img_path.name}",
                cv2.cvtColor(processed, cv2.COLOR_RGB2BGR)
            )
            count += 1
        print(f"   ✅ {cat}: {count} images → Saved to Drive!")
    print("✅ Preprocessing complete!")
else:
    print("✅ Data already on Drive — skipping!")

# ============================================
# Drive → Local copy (for training performance)
# ============================================
print("\n⏳ Copying from Drive to Local...")
for cat in ['benign', 'malignant']:
    local_dir = f"{PROCESSED}/{cat}"
    os.makedirs(local_dir, exist_ok=True)
    files = [f for f in os.listdir(
        f"{DRIVE_PROCESSED}/{cat}")
        if f.endswith('.png')]
    for f in files:
        shutil.copy(
            f"{DRIVE_PROCESSED}/{cat}/{f}",
            f"{local_dir}/{f}"
        )
print(f"✅ Local copy complete!")

# ============================================
# Dataset Split (Local)
# ============================================
def split_dataset(category, train_r=0.70, val_r=0.15):
    src    = f"{PROCESSED}/{category}"
    images = sorted([
        f for f in os.listdir(src) if f.endswith('.png')
    ])
    # Fixed seed for consistent splitting
    random.seed(RANDOM_SEED)
    random.shuffle(images)
    n     = len(images)
    t_end = int(n * train_r)
    v_end = int(n * (train_r + val_r))
    splits = {
        'train': images[:t_end],
        'val':   images[t_end:v_end],
        'test':  images[v_end:]
    }
    counts = {}
    for split, imgs in splits.items():
        dest = f"{DATASET_PATH}/{split}/{category}"
        os.makedirs(dest, exist_ok=True)
        for img in imgs:
            shutil.copy(f"{src}/{img}", f"{dest}/{img}")
        counts[split] = len(imgs)
    return counts

if not os.path.exists(f"{DATASET_PATH}/train/benign"):
    print("\n⏳ Splitting dataset...")
    b = split_dataset('benign')
    m = split_dataset('malignant')
    print(f"   Train: Benign={b['train']} Malignant={m['train']}")
    print(f"   Val:   Benign={b['val']}  Malignant={m['val']}")
    print(f"   Test:  Benign={b['test']} Malignant={m['test']}")
else:
    print("\n✅ Dataset split already exists — skipping!")

print("\n✅ CELL 2 complete — Proceed to the next cell!")

📊 Processed images on Drive:
   Benign:    437 units
   Malignant: 210 units
✅ Data already on Drive — skipping!

⏳ Copying from Drive to Local...
✅ Local copy complete!

⏳ Splitting dataset...
   Train: Benign=305 Malignant=147
   Val:   Benign=66  Malignant=31
   Test:  Benign=66 Malignant=32

✅ CELL 2 complete — Proceed to the next cell!


# **Cell 3 — Generators + Functions**


In [ ]:
# ============================================
# CELL 3 IMPROVED: GENERATORS + FUNCTIONS
# All models will be trained with improved settings
# ============================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import (
    InceptionV3, Xception, DenseNet121, VGG16, ResNet50
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)
import matplotlib.pyplot as plt
import json, gc

# ============================================
# Generators
# ============================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.20,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    fill_mode='nearest'
)
vt_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    f"{DATASET_PATH}/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=RANDOM_SEED
)
val_generator = vt_datagen.flow_from_directory(
    f"{DATASET_PATH}/val",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)
test_generator = vt_datagen.flow_from_directory(
    f"{DATASET_PATH}/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

labels        = train_generator.classes
b_n           = int(np.sum(labels == 0))
m_n           = int(np.sum(labels == 1))
total_n       = b_n + m_n
CLASS_WEIGHTS = {
    0: total_n / (2 * b_n),
    1: total_n / (2 * m_n)
}

print(f"✅ Train:{train_generator.samples} | "
      f"Val:{val_generator.samples} | "
      f"Test:{test_generator.samples}")
print(f"✅ Weights → Benign:{CLASS_WEIGHTS[0]:.3f} "
      f"Malignant:{CLASS_WEIGHTS[1]:.3f}")

# ============================================
# IMPROVED BUILD MODEL
# Optimized settings for each model
# ============================================
# Individual configurations for each architecture
MODEL_CONFIGS = {
    'InceptionV3': {
        'fe_freeze': 0.50,  # 50% freeze
        'ft_freeze': 0.40,  # 40% freeze
        'fe_lr':     1e-4,
        'ft_lr':     5e-6,
    },
    'Xception': {
        'fe_freeze': 0.50,
        'ft_freeze': 0.40,
        'fe_lr':     1e-4,
        'ft_lr':     5e-6,
    },
    'DenseNet121': {
        'fe_freeze': 0.50,
        'ft_freeze': 0.40,
        'fe_lr':     1e-4,
        'ft_lr':     5e-6,
    },
    'VGG16': {
        'fe_freeze': 0.60,  # VGG16 is large, freezing slightly more
        'ft_freeze': 0.50,
        'fe_lr':     1e-4,
        'ft_lr':     5e-6,
    },
    'ResNet50': {
        'fe_freeze': 0.50,
        'ft_freeze': 0.40,
        'fe_lr':     1e-4,
        'ft_lr':     5e-6,
    },
}

def build_model_improved(base_fn, model_name, strategy):
    config = MODEL_CONFIGS[model_name]

    base = base_fn(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # Freeze settings
    if strategy == 'feature_extractor':
        freeze_ratio = config['fe_freeze']
        lr           = config['fe_lr']
    else:
        freeze_ratio = config['ft_freeze']
        lr           = config['ft_lr']

    base.trainable = True
    freeze_until   = int(len(base.layers) * freeze_ratio)
    for layer in base.layers[:freeze_until]:
        layer.trainable = False
    for layer in base.layers[freeze_until:]:
        layer.trainable = True

    # Improved Classifier
    # Added BatchNorm for training stability
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)  # New
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base.input, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auroc')]
    )

    trainable = sum([
        tf.size(w).numpy()
        for w in model.trainable_weights
    ])
    total = sum([
        tf.size(w).numpy() for w in model.weights
    ])
    print(f"   Freeze:{freeze_ratio*100:.0f}% | "
          f"LR:{lr} | "
          f"Trainable:{trainable:,}/{total:,}")
    return model

# ============================================
# IMPROVED CALLBACKS
# ============================================
def get_callbacks(prefix):
    return [
        ModelCheckpoint(
            filepath=(f"{SAVE_DIR}/models/"
                      f"{prefix}_best.keras"),
            monitor='val_auroc',
            mode='max',
            save_best_only=True,
            verbose=1
        ),
        EarlyStopping(
            monitor='val_auroc',
            mode='max',
            patience=20,      # Increased from 15
            verbose=1,
            restore_best_weights=True
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,       # Reduced from 0.5
            patience=5,
            min_lr=1e-8,
            verbose=1
        )
    ]

# ============================================
# PLOT HISTORY
# ============================================
def plot_history(history, prefix):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f'Training History — {prefix}',
        fontsize=14, fontweight='bold'
    )
    for ax, (metric, title, color) in zip(axes, [
        ('loss',     'Loss',     'red'),
        ('accuracy', 'Accuracy', 'blue'),
        ('auroc',    'AUROC',    'green'),
    ]):
        ax.plot(history[metric], label='Train',
                color=color, linewidth=2)
        ax.plot(history[f'val_{metric}'], label='Val',
                color=color, linewidth=2, linestyle='--')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        f"{SAVE_DIR}/plots/history_{prefix}.png",
        dpi=150, bbox_inches='tight'
    )
    plt.close()

# ============================================
# TRAIN ONE MODEL
# ============================================
def train_one_model(model_name, base_fn,
                    strategy, epochs=50):
    prefix    = f"{model_name}_{strategy}"
    save_path = f"{SAVE_DIR}/models/{prefix}_best.keras"

    print(f"\n{'='*55}")
    print(f"🚀 {model_name} — {strategy}")
    print(f"{'='*55}")

    # Check if model already exists on Drive
    if os.path.exists(save_path):
        size = os.path.getsize(save_path)/1024/1024
        print(f"⏭️  Already on Drive ({size:.1f} MB) — skipping!")
        return

    model = build_model_improved(
        base_fn, model_name, strategy
    )
    train_generator.reset()
    val_generator.reset()

    history = model.fit(
        train_generator,
        epochs=epochs,
        validation_data=val_generator,
        callbacks=get_callbacks(prefix),
        class_weight=CLASS_WEIGHTS,
        verbose=1
    )

    hist_dict = {
        k: [float(v) for v in vals]
        for k, vals in history.history.items()
    }
    with open(
        f"{SAVE_DIR}/history/{prefix}_history.json",
        'w'
    ) as f:
        json.dump(hist_dict, f)

    plot_history(hist_dict, prefix)
    best_auroc = max(hist_dict['val_auroc'])

    if os.path.exists(save_path):
        size = os.path.getsize(save_path)/1024/1024
        print(f"\n✅ {prefix} complete!")
        print(f"   Best Val AUROC: {best_auroc:.4f}")
        print(f"   💾 Saved to Drive: {size:.1f} MB ✅")
    else:
        print(f"❌ Failed to save on Drive!")

    del model
    gc.collect()
    tf.keras.backend.clear_session()

print("\n✅ CELL 3 IMPROVED complete!")
print("   Run the next cell to start training!")

Found 452 images belonging to 2 classes.
Found 97 images belonging to 2 classes.
Found 98 images belonging to 2 classes.
✅ Train:452 | Val:97 | Test:98
✅ Weights → Benign:0.741 Malignant:1.537

✅ CELL 3 IMPROVED complete!
   Run the next cell to start training!


# **Cell 4 — Training Model**

In [ ]:
# ============================================
# CELL 4 IMPROVED: TRAINING
# Deletes old models from Drive and
# trains the new improved versions
# ============================================

EPOCHS = 50  # Increased from 30 to 50

training_queue = [
    ('InceptionV3', InceptionV3, 'feature_extractor'),
    ('InceptionV3', InceptionV3, 'finetune'),
    ('Xception',    Xception,    'feature_extractor'),
    ('Xception',    Xception,    'finetune'),
    ('DenseNet121', DenseNet121, 'feature_extractor'),
    ('DenseNet121', DenseNet121, 'finetune'),
    ('VGG16',       VGG16,       'feature_extractor'),
    ('VGG16',       VGG16,       'finetune'),
    ('ResNet50',    ResNet50,    'feature_extractor'),
    ('ResNet50',    ResNet50,    'finetune'),
]

# ============================================
# Delete existing models to ensure
# improved versions are trained from scratch
# ============================================
print("🗑️  Deleting previous models...")
models_dir = f"{SAVE_DIR}/models"
deleted = 0
for name, _, strategy in training_queue:
    prefix = f"{name}_{strategy}"
    path   = f"{models_dir}/{prefix}_best.keras"
    if os.path.exists(path):
        os.remove(path)
        print(f"   🗑️  {prefix} deleted!")
        deleted += 1
print(f"✅ {deleted} old models have been removed!")

# ============================================
# Display Training Status
# ============================================
print(f"\n📊 Training Status ({EPOCHS} epochs):")
for name, _, strategy in training_queue:
    prefix = f"{name}_{strategy}"
    path   = f"{SAVE_DIR}/models/{prefix}_best.keras"
    status = "✅ Exists" if os.path.exists(path) else "❌ To be trained"
    print(f"   {status} — {prefix}")

print(f"\n⚠️  Keep the browser console alive!")
print(f"⏳ Starting Improved Training...\n")

# ============================================
# Train all models in queue
# ============================================
for name, fn, strategy in training_queue:
    train_one_model(
        name, fn,
        strategy=strategy,
        epochs=EPOCHS
    )

# ============================================
# Final Summary
# ============================================
print("\n" + "="*55)
print("🎉 All Improved Training Sessions Complete!")
print("="*55)

done = [f for f in os.listdir(models_dir)
        if f.endswith('.keras')]
print(f"\n💾 Found {len(done)}/10 models on Drive:")
for f in sorted(done):
    size = os.path.getsize(f"{models_dir}/{f}")
    print(f"   ✅ {f} — {size/1024/1024:.1f} MB")

🗑️  Deleting previous models...
   🗑️  InceptionV3_feature_extractor deleted!
   🗑️  InceptionV3_finetune deleted!
   🗑️  Xception_feature_extractor deleted!
   🗑️  Xception_finetune deleted!
   🗑️  DenseNet121_feature_extractor deleted!
   🗑️  DenseNet121_finetune deleted!
✅ 6 old models have been removed!

📊 Training Status (50 epochs):
   ❌ To be trained — InceptionV3_feature_extractor
   ❌ To be trained — InceptionV3_finetune
   ❌ To be trained — Xception_feature_extractor
   ❌ To be trained — Xception_finetune
   ❌ To be trained — DenseNet121_feature_extractor
   ❌ To be trained — DenseNet121_finetune
   ❌ To be trained — VGG16_feature_extractor
   ❌ To be trained — VGG16_finetune
   ❌ To be trained — ResNet50_feature_extractor
   ❌ To be trained — ResNet50_finetune

⚠️  Keep the browser console alive!
⏳ Starting Improved Training...


🚀 InceptionV3 — feature_extractor
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
   Freeze:50% | LR:0.0001 | Trainable:17,971,137/22,985,505
Ep

In [ ]:
# ============================================
# RETRAIN 3 FINETUNE MODELS
# Xception, InceptionV3, ResNet50 finetune
# Previous LR (5e-6) was too small → use 1e-5
# ============================================

import os, gc, json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import (
    InceptionV3, Xception, ResNet50
)
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)

SAVE_DIR = "/content/drive/MyDrive/BreastCancerNew/results"

# ============================================
# DELETE OLD BAD MODELS
# ============================================
to_retrain = [
    ('Xception',    Xception,    'finetune', 1e-5),
    ('InceptionV3', InceptionV3, 'finetune', 1e-5),
    ('ResNet50',    ResNet50,    'finetune', 1e-5),
]

print("Deleting old models...")
for name, _, strategy, _ in to_retrain:
    prefix = f"{name}_{strategy}"
    path   = f"{SAVE_DIR}/models/{prefix}_best.keras"
    if os.path.exists(path):
        os.remove(path)
        print(f"   Deleted: {prefix}")

# ============================================
# BUILD MODEL FUNCTION
# feature_extractor: 50% freeze, LR 1e-4
# finetune: 40% freeze, LR 1e-5
# ============================================
def build_model_fixed(base_fn, strategy, lr):
    base = base_fn(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3)
    )

    # Set freeze ratio based on strategy
    base.trainable = True
    freeze_ratio   = (0.50 if strategy == 'feature_extractor'
                      else 0.40)
    freeze_until   = int(len(base.layers) * freeze_ratio)

    for layer in base.layers[:freeze_until]:
        layer.trainable = False
    for layer in base.layers[freeze_until:]:
        layer.trainable = True

    # Classifier with BatchNorm for stable training
    x      = base.output
    x      = layers.GlobalAveragePooling2D()(x)
    x      = layers.Dense(512, activation='relu')(x)
    x      = layers.BatchNormalization()(x)
    x      = layers.Dropout(0.4)(x)
    x      = layers.Dense(256, activation='relu')(x)
    x      = layers.Dropout(0.3)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base.input, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auroc')]
    )

    trainable = sum([
        tf.size(w).numpy()
        for w in model.trainable_weights
    ])
    print(f"   LR:{lr} | "
          f"Freeze:{freeze_ratio*100:.0f}% | "
          f"Trainable:{trainable:,}")
    return model

# ============================================
# CALLBACKS
# ============================================
def get_callbacks(prefix):
    return [
        # Save best model based on val_auroc
        ModelCheckpoint(
            filepath=(f"{SAVE_DIR}/models/"
                      f"{prefix}_best.keras"),
            monitor='val_auroc',
            mode='max',
            save_best_only=True,
            verbose=1
        ),
        # Stop if no improvement after 20 epochs
        EarlyStopping(
            monitor='val_auroc',
            mode='max',
            patience=20,
            verbose=1,
            restore_best_weights=True
        ),
        # Reduce LR if val_loss plateaus
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,
            patience=5,
            min_lr=1e-8,
            verbose=1
        )
    ]

# ============================================
# PLOT TRAINING HISTORY
# ============================================
def plot_history(history, prefix):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f'Training History — {prefix}',
        fontsize=14, fontweight='bold'
    )
    metrics = [
        ('loss',     'Loss',     'red'),
        ('accuracy', 'Accuracy', 'blue'),
        ('auroc',    'AUROC',    'green'),
    ]
    for ax, (metric, title, color) in zip(axes, metrics):
        ax.plot(history[metric],
                label='Train', color=color, linewidth=2)
        ax.plot(history[f'val_{metric}'],
                label='Val', color=color,
                linewidth=2, linestyle='--')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        f"{SAVE_DIR}/plots/history_{prefix}.png",
        dpi=150, bbox_inches='tight'
    )
    plt.close()

# ============================================
# TRAIN FUNCTION
# ============================================
def train_one_model(name, fn, strategy, lr, epochs=50):
    prefix    = f"{name}_{strategy}"
    save_path = f"{SAVE_DIR}/models/{prefix}_best.keras"

    print(f"\n{'='*50}")
    print(f"Training: {name} — {strategy}")
    print(f"{'='*50}")

    # Skip if already exists in Drive
    if os.path.exists(save_path):
        size = os.path.getsize(save_path)/1024/1024
        print(f"Already in Drive ({size:.1f}MB) — skip!")
        return

    model = build_model_fixed(fn, strategy, lr)
    train_generator.reset()
    val_generator.reset()

    history = model.fit(
        train_generator,
        epochs=epochs,
        validation_data=val_generator,
        callbacks=get_callbacks(prefix),
        class_weight=CLASS_WEIGHTS,
        verbose=1
    )

    # Save training history to JSON
    hist_dict = {
        k: [float(v) for v in vals]
        for k, vals in history.history.items()
    }
    with open(
        f"{SAVE_DIR}/history/{prefix}_history.json",
        'w'
    ) as f:
        json.dump(hist_dict, f)

    plot_history(hist_dict, prefix)
    best_auroc = max(hist_dict['val_auroc'])

    # Confirm save to Drive
    if os.path.exists(save_path):
        size = os.path.getsize(save_path)/1024/1024
        print(f"\nCompleted: {prefix}")
        print(f"   Best Val AUROC: {best_auroc:.4f}")
        print(f"   Saved to Drive: {size:.1f} MB")
    else:
        print(f"ERROR: Not saved to Drive!")

    del model
    gc.collect()
    tf.keras.backend.clear_session()

# ============================================
# RUN TRAINING
# ============================================
print("Retraining 3 finetune models...\n")

for name, fn, strategy, lr in to_retrain:
    train_one_model(name, fn, strategy, lr, epochs=50)

# ============================================
# FINAL DRIVE STATUS
# ============================================
print("\n" + "="*50)
print("Retraining Complete!")
print("="*50)

done = sorted([
    f for f in os.listdir(f"{SAVE_DIR}/models")
    if f.endswith('.keras')
])
print(f"\nDrive: {len(done)} models saved:")
for f in done:
    size = os.path.getsize(f"{SAVE_DIR}/models/{f}")
    print(f"   {f} — {size/1024/1024:.1f} MB")

Deleting old models...
   Deleted: Xception_finetune
   Deleted: InceptionV3_finetune
   Deleted: ResNet50_finetune
Retraining 3 finetune models...


Training: Xception — finetune
   LR:1e-05 | Freeze:40% | Trainable:18,191,929
Epoch 1/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6168 - auroc: 0.5918 - loss: 0.8262
Epoch 1: val_auroc improved from None to 0.74804, saving model to /content/drive/MyDrive/BreastCancerNew/results/models/Xception_finetune_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/BreastCancerNew/results/models/Xception_finetune_best.keras
15/15 ━━━━━━━━━━━━━━━━━━━━ 83s 3s/step - accuracy: 0.6416 - auroc: 0.6293 - loss: 0.7629 - val_accuracy: 0.6804 - val_auroc: 0.7480 - val_loss: 0.6270 - learning_rate: 1.0000e-05
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 568ms/step - accuracy: 0.6881 - auroc: 0.6984 - loss: 0.6655
Epoch 2: val_auroc improved from 0.74804 to 0.89785, saving model to /content/drive/MyDrive/BreastCancerNew/results/models/Xc

# **Cell 5 — Evaluation**

finetune model retrain

In [ ]:
# ============================================
# FINAL EVALUATION — Run immediately!
# ============================================

import pandas as pd, gc
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, roc_curve
)

SAVE_DIR = "/content/drive/MyDrive/BreastCancerNew/results"

training_queue = [
    ('InceptionV3', 'feature_extractor'),
    ('InceptionV3', 'finetune'),
    ('Xception',    'feature_extractor'),
    ('Xception',    'finetune'),
    ('DenseNet121', 'feature_extractor'),
    ('DenseNet121', 'finetune'),
    ('VGG16',       'feature_extractor'),
    ('VGG16',       'finetune'),
    ('ResNet50',    'feature_extractor'),
    ('ResNet50',    'finetune'),
]

results = []

for model_name, strategy in training_queue:
    prefix    = f"{model_name}_{strategy}"
    save_path = f"{SAVE_DIR}/models/{prefix}_best.keras"

    if not os.path.exists(save_path):
        print(f"MISSING: {prefix} — skipping!")
        continue

    print(f"Evaluating: {prefix}")
    model = tf.keras.models.load_model(save_path)

    test_generator.reset()
    y_pred_prob = model.predict(
        test_generator, verbose=0).flatten()
    y_true      = test_generator.classes
    y_pred      = (y_pred_prob >= 0.5).astype(int)

    # Metrics calculation
    auroc          = roc_auc_score(y_true, y_pred_prob)
    cm             = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    sensitivity    = tp / (tp + fn)
    specificity    = tn / (tn + fp)
    accuracy       = (tp + tn) / (tp + tn + fp + fn)
    ppv            = tp/(tp+fp) if (tp+fp) > 0 else 0
    npv            = tn/(tn+fn) if (tn+fn) > 0 else 0

    results.append({
        'Model':       model_name,
        'Strategy':    strategy,
        'AUROC':       round(auroc, 4),
        'Accuracy':    round(accuracy, 4),
        'Sensitivity': round(sensitivity, 4),
        'Specificity': round(specificity, 4),
        'PPV':         round(ppv, 4),
        'NPV':         round(npv, 4),
        'TP': int(tp), 'TN': int(tn),
        'FP': int(fp), 'FN': int(fn),
    })

    print(f"   AUROC:{auroc:.4f} | "
          f"Sen:{sensitivity:.4f} | "
          f"Spe:{specificity:.4f} | "
          f"Acc:{accuracy:.4f}")

    # Memory cleanup
    del model
    gc.collect()
    tf.keras.backend.clear_session()

# Save results to Drive as CSV
df = pd.DataFrame(results)
df = df.sort_values('AUROC', ascending=False
                    ).reset_index(drop=True)

print("\n" + "="*75)
print("FINAL RESULTS SUMMARY")
print("="*75)
print(df[['Model','Strategy','AUROC','Accuracy',
          'Sensitivity','Specificity','PPV','NPV']
         ].to_string(index=False))

df.to_csv(f"{SAVE_DIR}/final_results.csv", index=False)
print(f"\n✅ Evaluation report saved to: {SAVE_DIR}/final_results.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, gc, shutil
import tensorflow as tf
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, roc_curve
)

# ==============================================================================
# 1. DIRECTORY CONFIGURATION & INITIALIZATION
# ==============================================================================
# Define result and plot directories
SAVE_DIR = "/content/drive/MyDrive/BreastCancerNew/results"
PLOTS_DIR = f"{SAVE_DIR}/plots"

# Refresh the plots folder to ensure no stale data remains
if os.path.exists(PLOTS_DIR):
    shutil.rmtree(PLOTS_DIR)
    print(f"🗑️ Previous plots directory purged: {PLOTS_DIR}")

os.makedirs(PLOTS_DIR, exist_ok=True)

# List of architectures and training methodologies to be evaluated
training_queue = [
    ('InceptionV3', 'feature_extractor'), ('InceptionV3', 'finetune'),
    ('Xception',    'feature_extractor'), ('Xception',    'finetune'),
    ('DenseNet121', 'feature_extractor'), ('DenseNet121', 'finetune'),
    ('VGG16',       'feature_extractor'), ('VGG16',       'finetune'),
    ('ResNet50',    'feature_extractor'), ('ResNet50',    'finetune'),
]

results = []

# Initialize a global figure for multi-model ROC comparison
plt.figure(figsize=(12, 10))
colors = plt.cm.tab10(np.linspace(0, 1, len(training_queue)))

# ==============================================================================
# 2. EVALUATION PIPELINE: PERFORMANCE METRICS & VISUALIZATION
# ==============================================================================
for i, (model_name, strategy) in enumerate(training_queue):
    prefix = f"{model_name}_{strategy}"
    save_path = f"{SAVE_DIR}/models/{prefix}_best.keras"

    if not os.path.exists(save_path):
        print(f"❌ SKIPPING: {prefix} (Model file not found at {save_path})")
        continue

    print(f"⏳ Evaluating Model Performance: {prefix}")

    # Load pre-trained model and reset generator for consistent evaluation
    model = tf.keras.models.load_model(save_path)
    test_generator.reset()

    # Inference: Generate probabilistic predictions and thresholded labels
    y_pred_prob = model.predict(test_generator, verbose=0).flatten()
    y_true = test_generator.classes
    y_pred = (y_pred_prob >= 0.5).astype(int)

    # Statistical Metrics: Confusion Matrix Components & AUROC
    auroc = roc_auc_score(y_true, y_pred_prob)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Log results for comparative analysis
    results.append({
        'Model': model_name,
        'Strategy': strategy,
        'AUROC': round(auroc, 4),
        'Accuracy': round((tp+tn)/(tp+tn+fp+fn), 4),
        'Sensitivity': round(tp/(tp+fn), 4),
        'Specificity': round(tn/(tn+fp), 4),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    })

    # --- ROC CURVE GENERATION ---
    # Plot individual model ROC curve onto the master comparison plot
    fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'{prefix} (AUC={auroc:.3f})')

    # --- CONFUSION MATRIX GENERATION ---
    # Visualize classification performance for the specific model
    fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
                xticklabels=['Benign', 'Malignant'],
                yticklabels=['Benign', 'Malignant'],
                annot_kws={"size": 16, "weight": "bold"})
    ax_cm.set_title(f'Confusion Matrix: {prefix}')
    ax_cm.set_ylabel('Actual Label')
    ax_cm.set_xlabel('Predicted Label')
    plt.tight_layout()

    # Export CM plot and release memory
    fig_cm.savefig(f"{PLOTS_DIR}/CM_{prefix}.png", dpi=150)
    plt.close(fig_cm)

    # Clean up TensorFlow session and system memory
    del model
    gc.collect()
    tf.keras.backend.clear_session()

# ==============================================================================
# 3. FINAL AGGREGATION & DATA EXPORT
# ==============================================================================
# Finalize master ROC plot formatting
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Baseline (Random)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Comparative ROC Analysis — All Architectures', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=8)
plt.grid(True, alpha=0.3)
plt.savefig(f"{PLOTS_DIR}/combined_roc_curves.png", dpi=200, bbox_inches='tight')
plt.show()

# Export comprehensive summary to CSV, sorted by AUROC performance
df = pd.DataFrame(results).sort_values('AUROC', ascending=False)
df.to_csv(f"{SAVE_DIR}/final_evaluation_results.csv", index=False)

print("\n" + "="*70)
print("✅ PIPELINE EXECUTION SUCCESSFUL")
print(f"📂 Visualizations: {PLOTS_DIR}")
print(f"📄 Statistical Data: {SAVE_DIR}/final_evaluation_results.csv")
print("="*70)
print(df[['Model', 'Strategy', 'AUROC', 'Accuracy', 'Sensitivity', 'Specificity']].to_string(index=False))

# **Cell 6 — Grad-CAM++ jaw**

In [ ]:
# ============================================
# CELL 6: GRAD-CAM++ EXPLAINABILITY (XAI)
# Best 4 models based on final evaluation:
# 1. DenseNet121 feature_extractor (AUROC:1.0000, Acc:0.9898)
# 2. Xception finetune             (AUROC:0.9995, Acc:0.9796)
# 3. InceptionV3 feature_extractor (AUROC:1.0000, Acc:0.9592)
# 4. ResNet50 feature_extractor    (AUROC:0.9991, Acc:0.9796)
# ============================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tensorflow as tf
import cv2
import os
import random
import gc

TEST_DIR = "/content/BUSI_dataset/test"
random.seed(42)

os.makedirs(f"{SAVE_DIR}/plots/gradcam", exist_ok=True)

# ============================================
# 6.1 — GRAD-CAM++ FUNCTION
# ============================================
def make_gradcampp_heatmap(img_array, model,
                            last_conv_layer_name):
    # Split model into two parts:
    # Part 1: Input → Last Conv Layer
    # Part 2: Last Conv Layer → Output
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    # Three GradientTapes — hallmark of Grad-CAM++
    with tf.GradientTape() as tape2:
        with tf.GradientTape() as tape1:
            with tf.GradientTape() as tape0:
                conv_outputs, predictions = \
                    grad_model(img_array)
                pred_index    = tf.argmax(predictions[0])
                class_channel = predictions[:, pred_index]
            grads1 = tape0.gradient(
                class_channel, conv_outputs)
        grads2 = tape1.gradient(grads1, conv_outputs)
    grads3 = tape2.gradient(grads2, conv_outputs)

    # Calculate weights using higher-order derivatives
    global_sum  = tf.reduce_sum(
        conv_outputs, axis=(0, 1, 2))
    alpha_num   = grads2[0]
    alpha_denom = (2.0 * grads2[0] +
                   global_sum * grads3[0])
    alpha_denom = tf.where(
        tf.equal(alpha_denom, 0),
        tf.ones_like(alpha_denom),
        alpha_denom
    )
    alphas     = alpha_num / alpha_denom
    alpha_norm = alphas / (
        tf.reduce_sum(alphas, axis=(0, 1)) + 1e-8)
    weights    = tf.reduce_sum(
        alpha_norm * tf.nn.relu(grads1[0]),
        axis=(0, 1)
    )

    # Generate activation heatmap
    conv_out = conv_outputs[0]
    heatmap  = tf.reduce_sum(weights * conv_out, axis=-1)
    heatmap  = tf.nn.relu(heatmap)
    heatmap  = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    confidence = float(predictions[0][pred_index])
    return heatmap.numpy(), confidence, int(pred_index)

# ============================================
# 6.2 — OVERLAY FUNCTION
# ============================================
def overlay_heatmap(img_orig, heatmap, alpha=0.45):
    heatmap_resized = cv2.resize(
        heatmap, (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_CUBIC
    )
    heatmap_resized = np.clip(heatmap_resized, 0, 1)
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(
        heatmap_colored, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(
        heatmap_colored, cv2.COLOR_BGR2RGB)
    img_uint8 = np.uint8(255 * img_orig)
    overlaid  = cv2.addWeighted(
        img_uint8,       1 - alpha,
        heatmap_colored, alpha, 0
    )
    return overlaid

# ============================================
# 6.3 — Last Conv Layer for Each Architecture
# ============================================
LAST_CONV_LAYERS = {
    'InceptionV3':  'mixed10',
    'Xception':     'block14_sepconv2_act',
    'DenseNet121':  'relu',
    'VGG16':        'block5_conv3',
    'ResNet50':     'conv5_block3_out',
}

# ============================================
# 6.4 — SHOW SAVED IMAGE FROM DRIVE
# ============================================
def show_saved_image(save_file, prefix):
    img = mpimg.imread(save_file)
    fig, ax = plt.subplots(figsize=(20, 8))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(
        f'Grad-CAM++ — {prefix} (Loaded from Drive)',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()
    print(f"Displayed from Drive: {prefix}")

# ============================================
# 6.5 — VISUALIZATION FUNCTION
# Skip if already saved + model unchanged
# Always display saved image
# ============================================
def visualize_gradcampp(model_name, strategy,
                         n_samples=3):
    prefix     = f"{model_name}_{strategy}"
    model_path = f"{SAVE_DIR}/models/{prefix}_best.keras"
    save_file  = (f"{SAVE_DIR}/plots/gradcam/"
                  f"gradcampp_{prefix}.png")

    print(f"\n{'='*50}")
    print(f"Grad-CAM++: {prefix}")
    print(f"{'='*50}")

    # Skip if plot exists and model not updated
    if os.path.exists(save_file):
        plot_time  = os.path.getmtime(save_file)
        model_time = os.path.getmtime(model_path)
        if plot_time >= model_time:
            print(f"Already in Drive — loading saved image...")
            show_saved_image(save_file, prefix)
            return
        else:
            print(f"Model updated — regenerating...")

    # Generate new Grad-CAM++ plot
    conv_layer = LAST_CONV_LAYERS[model_name]
    model      = tf.keras.models.load_model(model_path)

    # 2 rows: Benign, Malignant
    # 3 cols per sample: Original, Heatmap, Overlay
    fig, axes = plt.subplots(
        2, n_samples * 3,
        figsize=(n_samples * 9, 9)
    )
    fig.suptitle(
        f'Grad-CAM++ Analysis — {model_name} ({strategy})\n'
        f'Red = High Model Focus | Blue = Area Ignored',
        fontsize=14, fontweight='bold', y=1.02
    )

    for row, (cat, true_label, color) in enumerate([
        ('benign',    0, 'green'),
        ('malignant', 1, 'red')
    ]):
        folder   = f"{TEST_DIR}/{cat}"
        all_imgs = [
            f for f in os.listdir(folder)
            if f.endswith('.png')
        ]
        imgs = random.sample(
            all_imgs, min(n_samples, len(all_imgs))
        )

        for i, img_name in enumerate(imgs):
            # Load and preprocess image
            img_path = os.path.join(folder, img_name)
            img_orig = cv2.imread(img_path)
            img_orig = cv2.cvtColor(
                img_orig, cv2.COLOR_BGR2RGB)
            img_orig = cv2.resize(
                img_orig, (IMG_SIZE, IMG_SIZE))
            img_orig = img_orig / 255.0

            img_array = np.expand_dims(
                img_orig, axis=0).astype(np.float32)

            # Generate Grad-CAM++ heatmap
            heatmap, confidence, pred_idx = \
                make_gradcampp_heatmap(
                    img_array, model, conv_layer)
            overlaid = overlay_heatmap(img_orig, heatmap)

            # Check prediction correctness
            pred_label = ("Malignant" if pred_idx == 1
                          else "Benign")
            conf_pct   = (confidence * 100
                          if pred_idx == 1
                          else (1-confidence) * 100)
            is_correct = (pred_idx == true_label)
            status     = "Correct" if is_correct \
                         else "Wrong"

            col_base = i * 3

            # Column 1: Original image
            axes[row, col_base].imshow(img_orig)
            axes[row, col_base].set_title(
                f'Original\n({cat.upper()})',
                color=color, fontsize=9,
                fontweight='bold'
            )
            axes[row, col_base].axis('off')

            # Column 2: Pure heatmap
            axes[row, col_base+1].imshow(
                heatmap, cmap='jet', vmin=0, vmax=1)
            axes[row, col_base+1].set_title(
                'Heatmap\nRed=Focus Blue=Ignore',
                fontsize=8
            )
            axes[row, col_base+1].axis('off')

            # Column 3: Blended overlay
            axes[row, col_base+2].imshow(overlaid)
            axes[row, col_base+2].set_title(
                f'{status}: {pred_label}\n'
                f'Conf: {conf_pct:.1f}%',
                fontsize=8,
                color='green' if is_correct else 'red'
            )
            axes[row, col_base+2].axis('off')

    plt.tight_layout()
    plt.savefig(save_file, dpi=150,
                bbox_inches='tight')
    plt.show()
    plt.close()

    del model
    gc.collect()
    tf.keras.backend.clear_session()
    print(f"Saved to Drive: gradcampp_{prefix}.png")

# ============================================
# 6.6 — BEST 4 MODELS FOR THESIS
# Selected based on final evaluation results:
# Criteria: High AUROC + High Accuracy + Balanced metrics
# ============================================
best_models = [
    ('DenseNet121', 'feature_extractor'),  # AUROC:1.0000 Acc:0.9898
    ('Xception',    'finetune'),            # AUROC:0.9995 Acc:0.9796
    ('InceptionV3', 'feature_extractor'),  # AUROC:1.0000 Acc:0.9592
    ('ResNet50',    'feature_extractor'),  # AUROC:0.9991 Acc:0.9796
]

# Show current Drive status
print("Drive Status — Grad-CAM++ plots:")
print("-" * 50)
for model_name, strategy in best_models:
    prefix    = f"{model_name}_{strategy}"
    save_file = (f"{SAVE_DIR}/plots/gradcam/"
                 f"gradcampp_{prefix}.png")
    if os.path.exists(save_file):
        size = os.path.getsize(save_file)/1024
        print(f"   EXISTS : {prefix} ({size:.0f} KB)")
    else:
        print(f"   MISSING: {prefix} — will generate")

print("\nStarting Grad-CAM++ process...\n")

for model_name, strategy in best_models:
    visualize_gradcampp(
        model_name,
        strategy=strategy,
        n_samples=3
    )

# Final summary
print("\n" + "="*50)
print("Grad-CAM++ Complete!")
print("="*50)
print("\nFinal Drive Status:")
gradcam_dir = f"{SAVE_DIR}/plots/gradcam"
files = sorted(os.listdir(gradcam_dir))
for f in files:
    size = os.path.getsize(f"{gradcam_dir}/{f}")
    print(f"   {f} — {size/1024:.0f} KB")

In [ ]:
import os, shutil, zipfile, cv2, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve

IMG_SIZE = 224
BATCH_SIZE = 32

DRIVE_BASE = "/content/drive/MyDrive/BreastCancerNew"
SAVE_DIR = f"{DRIVE_BASE}/results"

BUSBRA_ZIP = f"{DRIVE_BASE}/external_data/BUSBRA.zip"
BUSBRA_URL = "https://zenodo.org/api/records/8231412/files/BUSBRA.zip/content"

BUSBRA_EXTRACT_DIR = "/content/BUSBRA_raw"
BUSBRA_ROOT = f"{BUSBRA_EXTRACT_DIR}/BUSBRA"

EXTERNAL_PROCESSED_DRIVE = f"{DRIVE_BASE}/external_processed/BUS_BRA"
EXTERNAL_PROCESSED_LOCAL = "/content/BUS_BRA_external_processed"

os.makedirs(f"{DRIVE_BASE}/external_data", exist_ok=True)
os.makedirs(EXTERNAL_PROCESSED_DRIVE, exist_ok=True)

print("Setup ready.")

In [ ]:
if not os.path.exists(BUSBRA_ZIP):
    print("Downloading BUS-BRA...")
    !wget -O "{BUSBRA_ZIP}" "{BUSBRA_URL}"
else:
    print("BUS-BRA zip already exists.")

if not os.path.exists(BUSBRA_ROOT):
    print("Extracting BUS-BRA...")
    os.makedirs(BUSBRA_EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(BUSBRA_ZIP, "r") as z:
        z.extractall(BUSBRA_EXTRACT_DIR)
else:
    print("BUS-BRA already extracted.")

print(os.listdir(BUSBRA_ROOT))

In [ ]:
df_meta = pd.read_csv(f"{BUSBRA_ROOT}/bus_data.csv")

print(df_meta.head())
print(df_meta.columns.tolist())
print(df_meta["Pathology"].value_counts())

In [ ]:
def preprocess_image_with_mask(img_path, mask_path, img_size=224):
    img = cv2.imread(str(img_path))
    if img is None:
        return None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None

    mask = cv2.resize(mask, (img.shape[1], img.shape[0]))

    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    masked_img = img.copy()
    masked_img[binary_mask == 0] = 0

    coords = cv2.findNonZero(binary_mask)

    if coords is None:
        cropped = masked_img
    else:
        x, y, w, h = cv2.boundingRect(coords)

        pad_x = int(w * 0.05)
        pad_y = int(h * 0.05)

        x1 = max(0, x - pad_x)
        y1 = max(0, y - pad_y)
        x2 = min(img.shape[1], x + w + pad_x)
        y2 = min(img.shape[0], y + h + pad_y)

        cropped = masked_img[y1:y2, x1:x2]

    h, w = cropped.shape[:2]

    if h != w:
        max_dim = max(h, w)
        square = np.zeros((max_dim, max_dim, 3), dtype=np.uint8)

        y_s = (max_dim - h) // 2
        x_s = (max_dim - w) // 2

        square[y_s:y_s+h, x_s:x_s+w] = cropped
    else:
        square = cropped

    return cv2.resize(square, (img_size, img_size))


img_dir = Path(f"{BUSBRA_ROOT}/Images")
mask_dir = Path(f"{BUSBRA_ROOT}/Masks")

for cls in ["benign", "malignant"]:
    os.makedirs(f"{EXTERNAL_PROCESSED_DRIVE}/{cls}", exist_ok=True)

processed_count = {"benign": 0, "malignant": 0}
missing_count = 0

for _, row in df_meta.iterrows():
    img_id = str(row["ID"])
    pathology = str(row["Pathology"]).strip().lower()

    if pathology not in ["benign", "malignant"]:
        continue

    img_path = img_dir / f"{img_id}.png"
    mask_id = img_id.replace("bus_", "mask_")
    mask_path = mask_dir / f"{mask_id}.png"

    if not img_path.exists() or not mask_path.exists():
        missing_count += 1
        continue

    processed = preprocess_image_with_mask(img_path, mask_path, IMG_SIZE)

    if processed is None:
        missing_count += 1
        continue

    save_path = f"{EXTERNAL_PROCESSED_DRIVE}/{pathology}/{img_id}.png"

    cv2.imwrite(
        save_path,
        cv2.cvtColor(processed, cv2.COLOR_RGB2BGR)
    )

    processed_count[pathology] += 1

print("Preprocessing complete.")
print(processed_count)
print("Missing/skipped:", missing_count)

In [ ]:
if os.path.exists(EXTERNAL_PROCESSED_LOCAL):
    shutil.rmtree(EXTERNAL_PROCESSED_LOCAL)

for cls in ["benign", "malignant"]:
    src = f"{EXTERNAL_PROCESSED_DRIVE}/{cls}"
    dst = f"{EXTERNAL_PROCESSED_LOCAL}/{cls}"
    os.makedirs(dst, exist_ok=True)

    for f in os.listdir(src):
        if f.endswith(".png"):
            shutil.copy(f"{src}/{f}", f"{dst}/{f}")

external_datagen = ImageDataGenerator(rescale=1./255)

external_generator = external_datagen.flow_from_directory(
    EXTERNAL_PROCESSED_LOCAL,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    classes=["benign", "malignant"],
    shuffle=False
)

print(external_generator.class_indices)
print("External samples:", external_generator.samples)

In [ ]:
EXTERNAL_RESULTS_DIR = f"{SAVE_DIR}/external_validation_BUS_BRA"
EXTERNAL_PLOTS_DIR = f"{EXTERNAL_RESULTS_DIR}/plots"

os.makedirs(EXTERNAL_RESULTS_DIR, exist_ok=True)
os.makedirs(EXTERNAL_PLOTS_DIR, exist_ok=True)

# training_queue = [
#     ('InceptionV3', 'feature_extractor'),
#     ('InceptionV3', 'finetune'),
#     ('Xception',    'feature_extractor'),
#     ('Xception',    'finetune'),
#     ('DenseNet121', 'feature_extractor'),
#     ('DenseNet121', 'finetune'),
#     ('VGG16',       'feature_extractor'),
#     ('VGG16',       'finetune'),
#     ('ResNet50',    'feature_extractor'),
#     ('ResNet50',    'finetune'),
# ]
training_queue = [
    ("DenseNet121", "feature_extractor"),
    ("DenseNet121", "finetune")
]

results = []

plt.figure(figsize=(12, 10))
colors = plt.cm.tab10(np.linspace(0, 1, len(training_queue)))

for i, (model_name, strategy) in enumerate(training_queue):
    prefix = f"{model_name}_{strategy}"
    model_path = f"{SAVE_DIR}/models/{prefix}_best.keras"

    if not os.path.exists(model_path):
        print(f"Missing model: {prefix}")
        continue

    print(f"External validating: {prefix}")

    model = tf.keras.models.load_model(model_path)

    external_generator.reset()

    raw_pred = model.predict(external_generator, verbose=0)

    if raw_pred.shape[-1] == 2:
        y_pred_prob = raw_pred[:, 1]
    else:
        y_pred_prob = raw_pred.flatten()

    y_true = external_generator.classes
    y_pred = (y_pred_prob >= 0.5).astype(int)

    auroc = roc_auc_score(y_true, y_pred_prob)
    cm = confusion_matrix(y_true, y_pred)

    tn, fp, fn, tp = cm.ravel()

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1 = 2 * ppv * sensitivity / (ppv + sensitivity) if (ppv + sensitivity) > 0 else 0

    results.append({
        "Model": model_name,
        "Strategy": strategy,
        "AUROC": round(auroc, 4),
        "Accuracy": round(accuracy, 4),
        "Sensitivity": round(sensitivity, 4),
        "Specificity": round(specificity, 4),
        "PPV": round(ppv, 4),
        "NPV": round(npv, 4),
        "F1": round(f1, 4),
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn)
    })

    print(
        f"AUROC={auroc:.4f} | "
        f"Acc={accuracy:.4f} | "
        f"Sen={sensitivity:.4f} | "
        f"Spe={specificity:.4f}"
    )

    fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
    plt.plot(
        fpr,
        tpr,
        color=colors[i],
        lw=2,
        label=f"{prefix} AUC={auroc:.3f}"
    )

    fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Benign", "Malignant"],
        yticklabels=["Benign", "Malignant"],
        ax=ax_cm
    )
    ax_cm.set_title(f"External CM: {prefix}")
    ax_cm.set_xlabel("Predicted Label")
    ax_cm.set_ylabel("Actual Label")
    plt.tight_layout()
    fig_cm.savefig(f"{EXTERNAL_PLOTS_DIR}/EXT_CM_{prefix}.png", dpi=150)
    plt.close(fig_cm)

    del model
    gc.collect()
    tf.keras.backend.clear_session()

plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("External Validation ROC — BUS-BRA")
plt.legend(loc="lower right", fontsize=8)
plt.grid(True, alpha=0.3)
plt.savefig(
    f"{EXTERNAL_PLOTS_DIR}/external_combined_roc_BUS_BRA.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

df_external = pd.DataFrame(results)
df_external = df_external.sort_values(
    ["Accuracy", "Sensitivity", "Specificity", "AUROC"],
    ascending=False
).reset_index(drop=True)

save_csv = f"{EXTERNAL_RESULTS_DIR}/external_validation_results_BUS_BRA.csv"
df_external.to_csv(save_csv, index=False)

print(df_external)
print("Saved:", save_csv)

In [ ]:
import os, gc
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix
)

FAST_RESULTS_DIR = f"{SAVE_DIR}/external_validation_BUS_BRA"
os.makedirs(FAST_RESULTS_DIR, exist_ok=True)

fast_models = [
    ("DenseNet121", "feature_extractor"),
    ("DenseNet121", "finetune"),
]

fast_results = []

# Load true labels once
external_generator.reset()
y_true = external_generator.classes

for model_name, strategy in fast_models:
    prefix = f"{model_name}_{strategy}"
    model_path = f"{SAVE_DIR}/models/{prefix}_best.keras"

    if not os.path.exists(model_path):
        print(f"Missing model: {prefix}")
        continue

    print(f"\nFast external validating: {prefix}")

    model = tf.keras.models.load_model(model_path)

    external_generator.reset()

    raw_pred = model.predict(
        external_generator,
        verbose=1
    )

    if raw_pred.shape[-1] == 2:
        y_pred_prob = raw_pred[:, 1]
    else:
        y_pred_prob = raw_pred.flatten()

    y_pred = (y_pred_prob >= 0.5).astype(int)

    auroc = roc_auc_score(y_true, y_pred_prob)
    cm = confusion_matrix(y_true, y_pred)

    tn, fp, fn, tp = cm.ravel()

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1 = 2 * ppv * sensitivity / (ppv + sensitivity) if (ppv + sensitivity) > 0 else 0

    fast_results.append({
        "Model": model_name,
        "Strategy": strategy,
        "AUROC": round(auroc, 4),
        "Accuracy": round(accuracy, 4),
        "Sensitivity": round(sensitivity, 4),
        "Specificity": round(specificity, 4),
        "PPV": round(ppv, 4),
        "NPV": round(npv, 4),
        "F1": round(f1, 4),
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn)
    })

    print(
        f"AUROC={auroc:.4f} | "
        f"Acc={accuracy:.4f} | "
        f"Sen={sensitivity:.4f} | "
        f"Spe={specificity:.4f} | "
        f"F1={f1:.4f}"
    )

    del model
    gc.collect()
    tf.keras.backend.clear_session()

df_fast_external = pd.DataFrame(fast_results)
df_fast_external = df_fast_external.sort_values(
    ["Accuracy", "Sensitivity", "Specificity", "AUROC"],
    ascending=False
).reset_index(drop=True)

save_path = f"{FAST_RESULTS_DIR}/fast_external_validation_BUS_BRA.csv"
df_fast_external.to_csv(save_path, index=False)

print("\nFAST EXTERNAL VALIDATION RESULTS")
print(df_fast_external)
print("\nSaved:", save_path)

In [ ]:
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/BreastCancerNew/results"
csv_path = f"{SAVE_DIR}/final_results.csv"

df = pd.read_csv(csv_path)

print("\n" + "="*90)
print("ALL THESIS MODEL RESULTS")
print("="*90)

print(df.to_string(index=False))

print("\n" + "="*90)
print("BEST MODEL")
print("="*90)

best = df.sort_values("AUROC", ascending=False).iloc[0]
print(best.to_string())

print("\n" + "="*90)
print("RANKED SUMMARY")
print("="*90)

summary_cols = [
    "Model", "Strategy", "AUROC", "Accuracy",
    "Sensitivity", "Specificity", "PPV", "NPV",
    "TP", "TN", "FP", "FN"
]

print(df[summary_cols].sort_values("AUROC", ascending=False).to_string(index=False))

In [ ]:
# ============================================
# RECREATE TEST GENERATOR
# ============================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 32
DATASET_PATH = "/content/BUSI_dataset"

vt_datagen = ImageDataGenerator(rescale=1./255)

test_generator = vt_datagen.flow_from_directory(
    f"{DATASET_PATH}/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"✅ Test samples: {test_generator.samples}")
print(f"✅ Classes: {test_generator.class_indices}")

# Ensamble

In [ ]:
# ============================================
# ENSEMBLE EVALUATION — BEST MODELS
# ============================================

import os, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score
)

SAVE_DIR = "/content/drive/MyDrive/BreastCancerNew/results"

best_models = [
    "DenseNet121_finetune",
    "Xception_finetune",
    "ResNet50_feature_extractor"
]

all_probs = []
y_true = test_generator.classes

for model_prefix in best_models:
    model_path = f"{SAVE_DIR}/models/{model_prefix}_best.keras"

    if not os.path.exists(model_path):
        print(f"❌ Missing model: {model_prefix}")
        continue

    print(f"✅ Loading: {model_prefix}")
    model = tf.keras.models.load_model(model_path)

    test_generator.reset()
    probs = model.predict(test_generator, verbose=1).flatten()
    all_probs.append(probs)

    del model
    gc.collect()
    tf.keras.backend.clear_session()

# Soft voting ensemble
ensemble_prob = np.mean(all_probs, axis=0)

# Best threshold using Youden's J
fpr, tpr, thresholds = roc_curve(y_true, ensemble_prob)
best_idx = np.argmax(tpr - fpr)
best_threshold = thresholds[best_idx]

y_pred = (ensemble_prob >= best_threshold).astype(int)

# Metrics
auroc = roc_auc_score(y_true, ensemble_prob)
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_true, y_pred)
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
precision = precision_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print("\n" + "="*70)
print("BEST MODEL ENSEMBLE RESULT")
print("="*70)

print(f"Models Used: {best_models}")
print(f"Best Threshold: {best_threshold:.4f}")
print(f"AUROC:       {auroc:.4f}")
print(f"Accuracy:    {accuracy:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"NPV:         {npv:.4f}")
print(f"F1-score:    {f1:.4f}")

print("\nConfusion Matrix:")
print(f"TN: {tn} | FP: {fp}")
print(f"FN: {fn} | TP: {tp}")

# Save result
ensemble_result = pd.DataFrame([{
    "Model": "Ensemble",
    "Models Used": ", ".join(best_models),
    "Threshold": round(best_threshold, 4),
    "AUROC": round(auroc, 4),
    "Accuracy": round(accuracy, 4),
    "Sensitivity": round(sensitivity, 4),
    "Specificity": round(specificity, 4),
    "Precision": round(precision, 4),
    "NPV": round(npv, 4),
    "F1": round(f1, 4),
    "TP": int(tp),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
}])

save_path = f"{SAVE_DIR}/ensemble_result.csv"
ensemble_result.to_csv(save_path, index=False)

print(f"\n✅ Ensemble result saved: {save_path}")